This notebook is the actual machine learning example, taking the pickle file from the train_biomass_model.ipynb notebook and applies it to actual satellite imagery from Google Earth Engine (GEE) to create a biomass map.

Here is how it works in plain English:

1. Connecting to the Source (GEE)
The script starts by reaching out to Google Earth Engine. It defines exactly what data it wants:

Where: A specific polygon (Area of Interest) in California.
When: All Landsat 8 images captured between 2015 and 2019.
What: The seven spectral bands (SR_B1 through SR_B7) that the model expects.

2. The Prediction UDF (predict_biomass)
This is the custom logic (UDF) that will run on every piece of the satellite map:

It receives a "chunk" of the satellite data as a table.
It checks for missing data (NaNs) so it doesn't waste time trying to predict "empty" pixels (like those obscured by clouds).
It feeds the valid pixels into your loaded_model to generate a biomass prediction.
It returns only the "Predicted Biomass" column.

3. Run the Computation!

robustraster:

Breaks the massive area into smaller 256x256 chunks.
It sets up the Dask workers to work on different chunks simulteneously.
It automatically sends your trained model to each worker so they all know how to make predictions.
As the workers finish, it gathers the results and stitches them back together into a GeoTIFF file.

In [ ]:
# 1. Import the Google Earth Engine libraries.
#    Authenticate to Google Earth Engine. This
#    step only needs to be done once.

import ee

ee.Authenticate()

In [ ]:
# 2. Define a small Area of Interest (AOI)

aoi = ee.Geometry.Polygon(
    [[[-120.5, 38.5],
      [-120.5, 38.55],
      [-120.4, 38.55],
      [-120.4, 38.5],
      [-120.5, 38.5]]]
)

In [ ]:
# 3. Create the Google Earth Engine Image Collection
start_date = '2015-01-01'
end_date = '2019-12-31'

def prep_sr_l8(image):
    # Apply the scaling factors to the optical bands
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)

    # Replace the original bands with the scaled ones.
    # Note: Cloud masks are commented out to mirror your example, 
    # but normally you'd want them for clean surface reflectance.
    return image.addBands(optical_bands, None, True)

# Get Landsat 8 and select all 7 surface reflectance bands
ic = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate(start_date, end_date)
      .map(prep_sr_l8)
      .select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']))

In [ ]:
# 4. Import necesssary libraries and Write the custom user-defined function (UDF).

import pandas as pd
import numpy as np
import joblib
import os

def predict_biomass(df, ml_model):
    """
    Takes a chunk of data (as a pandas DataFrame) and a pre-trained scikit-learn model.
    Predicts biomass using Landsat 8 surface reflectance bands.
    """
    df = df.copy()
    bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    
    # We only want to predict where we have valid data for all required bands
    valid_mask = df[bands].notna().all(axis=1)
    
    # Initialize the output column with NaNs
    df['predicted_biomass'] = np.nan
    
    # If there is any valid data in this chunk, run the prediction
    if valid_mask.any():
        X = df.loc[valid_mask, bands]
        predictions = ml_model.predict(X)
        df.loc[valid_mask, 'predicted_biomass'] = predictions
        
    # Return a DataFrame with just the new output column
    return df[['predicted_biomass']]


In [ ]:
# 5 Load the pre-trained model derived from train_biomass_model.ipynb

model_path = "biomass_model.pkl"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Could not find '{model_path}'. "
        "Please run the code from 'train_biomass_model.ipynb' first to generate the random model."
    )
        
print(f"Loading pre-trained model from {model_path}...")

# Load the model into memory. Dask will serialize this object and distribute it to workers automatically.
loaded_model = joblib.load(model_path)

In [ ]:
# 6. Import the run() function from the robustraster package and do the computation!
from robustraster import run

print("Starting Machine Learning prediction on Landsat imagery...")
    
# Time chunking at -1 is usually a safe default for operations that might want to look at
# the time series, but for simple point-in-time predictions like this one, it can be chunked differently if memory is an issue.
# We will stick to -1 to be consistent with previous demos.
chunks = {"time": -1, "X": 256, "Y": 256}
    
run(
    dataset=ic,
    dataset_config={
        'geometry': aoi,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": predict_biomass,
        "user_function_args": (),
        # We pass the loaded model object via kwargs.
        "user_function_kwargs": {"ml_model": loaded_model},
    },
    function_tuning_config={
        "chunks": chunks,
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "ML_Prediction_Output",
        "vrt": True,
        "report": True
    }
)